In [ ]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#load dataset
df = pd.read_csv('crop_yield.csv')
df = df.sample(frac=0.03)

In [ ]:
#data cleaning
print(f"precleaning shape: {df.shape}")
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
print(f"postcleaning shape: {df.shape}\n")

In [ ]:
plt.figure(figsize=(15, 10))
#distributions
cols_to_plot = ['Rainfall_mm', 'Temperature_Celsius', 'Yield_tons_per_hectare']
for i, col in enumerate(cols_to_plot, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')


plt.subplot(2, 3, 4)
sns.boxplot(x='Crop', y='Yield_tons_per_hectare', data=df)
plt.title('Yield Variation by Crop Type')
plt.xticks(rotation=45)


plt.subplot(2, 3, 5)
sns.boxplot(x='Soil_Type', y='Yield_tons_per_hectare', data=df)
plt.title('Yield Variation by Soil Type')

#heatmap
plt.subplot(2, 3, 6)
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='mako')
plt.title('Correlation Heatmap')

plt.tight_layout()
plt.show()

In [ ]:
#preprocessing
X = df.drop('Yield_tons_per_hectare', axis=1)
y = df['Yield_tons_per_hectare']

numeric_features = ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest']
categorical_features = ['Region', 'Soil_Type', 'Crop', 'Weather_Condition', 'Fertilizer_Used', 'Irrigation_Used']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)


In [ ]:
#model
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=19)
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
#earlystopping training
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=100, batch_size=64, 
                    validation_split=0.2, callbacks=[early_stop], verbose=1)

In [ ]:
#visualisation
y_pred = model.predict(X_test).flatten()
residuals = y_test - y_pred

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.4, color='darkgreen')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title(f'Actual vs Predicted (R2: {r2_score(y_test, y_pred):.3f})')
plt.xlabel('Actual Yield')
plt.ylabel('Predicted Yield')

plt.subplot(1, 2, 2)
sns.histplot(residuals, kde=True, color='purple')
plt.title('Residuals (Error) Distribution')
plt.xlabel('Prediction Error')

plt.tight_layout()
plt.show()